In [9]:
import json
import pandas as pd
import numpy as np
import os

# 指定文件夹路径
folder_path = '/home/linkco/exa/Useful-Embedding/data/analyze/'

# 获取所有文件和文件夹名称
files = os.listdir(folder_path)

# 筛选出以.json结尾的文件
json_files = [f for f in files if f.endswith('.json')]

for file in json_files[:1]:
    model_name = file.replace(".json", "")
    print(model_name)
    # 读取文件
    file_path = folder_path + "{}.json".format(model_name)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 提取任务信息
    tasks = data["task_name"]

    model_dim = data["model_dim"]

    fused_weight_list = []
    for task_name, task_data in tasks.items():
        
        fused_weight = np.zeros(model_dim)
        # print('【 task_name 】', task_name)

        for split_win_size in data["task_name"][task_name]["split_win_size"].keys():
            # print('【 split_win_size 】', split_win_size)
            chunk_result = data["task_name"][task_name]["split_win_size"][split_win_size]["chunk_result"]
            sorted_index_list = [index for index, value in sorted(enumerate(chunk_result), key=lambda x: x[1], reverse=True)]
            # print('【 sorted_index_list 】', len(sorted_index_list), sorted_index_list)

            for index in sorted_index_list[:int(len(sorted_index_list)/2)]:
                fused_weight[index*int(split_win_size): (index+1)*int(split_win_size)] += 1
        
        # print(fused_weight)
        # print('【 】', )
        fused_weight /= len(data["task_name"][task_name]["split_win_size"].keys())
        fused_weight_list.append(fused_weight)
    
    # 计算各个维度根据不同任务，是否存在权重差异的使用上的不同，方差
    # 将四个数组按列堆叠，形状变为 (4, n)
    stacked = np.vstack(np.array(fused_weight_list))

    # 沿着 axis=0 计算每列的方差
    variances = np.var(stacked, axis=0)

    # 对所有位置的方差求平均
    mean_variance = np.mean(variances)

    # print("每个索引位置的方差:", variances)
    print("平均方差:", mean_variance)






stella_en_400M_v5
[0.28571429 0.28571429 0.28571429 ... 0.57142857 0.57142857 0.57142857]
[0.85714286 0.85714286 0.85714286 ... 0.14285714 0.14285714 0.14285714]
[0.42857143 0.42857143 0.42857143 ... 0.57142857 0.57142857 0.57142857]
[0.14285714 0.14285714 0.14285714 ... 0.28571429 0.28571429 0.28571429]
